# kaggle-002-Z-Image-Turbo-GGUF

**目标：Kaggle 2×T4，Z-Image-Turbo GGUF，双卡并行常驻推理。**

沿用 `kaggle-001-SANA-Sprint-1.6B` 的 v2 架构：

```text
Windows FastAPI / Cloudflare Tunnel
              │
              │ GET /task/next
              ▼
        Kaggle Task Queue
          ┌────┴────┐
          ▼         ▼
     GPU0 Worker  GPU1 Worker
          │         │
     sd-server   sd-server
     :12340      :12341
          │         │
          └────┬────┘
               ▼
         Upload Queue
               │
       WebP → AES-GCM
               │ POST /upload
               ▼
       Windows FastAPI
```

关键区别：

- **不是** `2×T4` 合起来跑一个模型。
- 每张 T4 各常驻一个 `stable-diffusion.cpp` `sd-server`，避免每个 Prompt 重新加载 GGUF。
- 默认：
  - diffusion: `z_image_turbo-Q4_K.gguf`
  - text encoder: `Qwen3-4B-Instruct-2507-Q4_K_M.gguf`
  - VAE: `ae.safetensors`
  - Z-Image-Turbo: `8 steps`, CFG `1.0`
- 继续使用原来的本地 FastAPI v2 协议：
  - `/task/next`
  - `/upload`
- 原 SANA UI 默认 `steps=2`，因此本 notebook **默认强制 Z-Image-Turbo 使用 8 steps**。如需让本地 UI 控制步数，再把 `USE_REMOTE_STEPS=True`。

参考：
- Tongyi-MAI / Z-Image
- leejet / stable-diffusion.cpp `docs/z_image.md`
- stable-diffusion.cpp Server API


## 0. GPU / Kaggle 环境检查


In [1]:
import os, sys, subprocess, platform, shutil

print("Python:", sys.version)
print("Platform:", platform.platform())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
subprocess.run(["nvidia-smi"], check=False)

try:
    import torch
    print("torch:", torch.__version__)
    print("cuda devices:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU{i}: {p.name} | {p.total_memory/1024**3:.2f} GiB | cc {p.major}.{p.minor}")
except Exception as e:
    print("torch GPU check:", e)

assert shutil.which("cmake"), "Kaggle 环境中未找到 cmake"
assert shutil.which("git"), "Kaggle 环境中未找到 git"
assert shutil.which("nvcc"), "未找到 nvcc；请确认 Kaggle Notebook 已开启 GPU"


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
CUDA_VISIBLE_DEVICES: None
Wed Aug 19 08:30:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|      

## 1. 可选：彻底清 GPU

如果前面已经加载过 Diffusers / SANA 等大模型，最可靠的方法仍然是结束当前 kernel，再重新运行 notebook。

**不要在正常运行 Dispatcher 时执行。**


In [2]:
# import os
# os._exit(0)


## 2. 安装 Python 依赖

使用 Hugging Face Hub 下载三个必要文件；网络层继续使用 `aiohttp`，图片加密继续使用 `cryptography`。


In [3]:
import subprocess, sys, os

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "huggingface_hub[hf_xet]", "aiohttp", "cryptography", "pillow", "requests"
], check=True)

# Hugging Face Xet 当前推荐的高吞吐开关
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

print("✓ Python dependencies ready")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 113.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 44.8 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
sigstore 4.3.0 requires cryptography<49,>=42, but you have cryptography 50.0.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you hav

✓ Python dependencies ready


## 3. 拉取并编译 stable-diffusion.cpp（CUDA / T4 sm_75）

只需要做一次。为 T4 指定 `CMAKE_CUDA_ARCHITECTURES=75`。

这里不需要它自己的 Web UI；我们只使用 `sd-server` HTTP API，所以关闭内置 frontend / WebP / WebM 构建项，减少不必要依赖。


In [ ]:
from pathlib import Path
import hashlib
import os
import shutil
import subprocess
import tarfile

WORK = Path("/kaggle/working/zimage_gguf")
PREBUILT = WORK / "stable-diffusion.cpp-prebuilt"

TAG = "sdcpp-t4-sm75-de298c225bed"
ASSET_NAME = "stable-diffusion-cpp-linux-cuda-t4-sm75-de298c225bed.tar.gz"
EXPECTED_SHA256 = "00075e290f3384ab08622ad4070476e9a4b4283bb8eb128d1e8248271106d30b"

DOWNLOAD_URL = (
    f"https://github.com/xiaoqianran/kaggle-build/"
    f"releases/download/{TAG}/{ASSET_NAME}"
)

ARCHIVE = WORK / ASSET_NAME
WORK.mkdir(parents=True, exist_ok=True)


def sha256_file(path: Path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with path.open("rb") as f:
        while chunk := f.read(chunk_size):
            h.update(chunk)
    return h.hexdigest()


# 1. 下载预编译 T4 / sm_75 构建
need_download = True

if ARCHIVE.exists():
    current_sha256 = sha256_file(ARCHIVE)
    if current_sha256 == EXPECTED_SHA256:
        print("✓ 已存在正确的预编译包，跳过下载")
        need_download = False
    else:
        print("旧文件 SHA256 不匹配，重新下载")
        ARCHIVE.unlink()

if need_download:
    print("Downloading:")
    print(DOWNLOAD_URL)

    subprocess.run([
        "curl",
        "-L",
        "--fail",
        "--retry", "3",
        "--retry-delay", "2",
        "-o", str(ARCHIVE),
        DOWNLOAD_URL,
    ], check=True)

# 2. SHA256 校验
actual_sha256 = sha256_file(ARCHIVE)

assert actual_sha256 == EXPECTED_SHA256, (
    "SHA256 校验失败\n"
    f"expected: {EXPECTED_SHA256}\n"
    f"actual:   {actual_sha256}"
)

print("✓ SHA256:", actual_sha256)

# 3. 解压
if PREBUILT.exists():
    shutil.rmtree(PREBUILT)

PREBUILT.mkdir(parents=True, exist_ok=True)

with tarfile.open(ARCHIVE, "r:gz") as tar:
    tar.extractall(PREBUILT)

print("✓ extracted:", PREBUILT)

# 4. 找 sd-server
candidates = [
    p for p in PREBUILT.rglob("sd-server")
    if p.is_file()
]

assert candidates, f"预编译包中未找到 sd-server：{PREBUILT}"

server_path = sorted(
    candidates,
    key=lambda p: len(str(p))
)[0]

server_path.chmod(server_path.stat().st_mode | 0o111)
SD_SERVER = str(server_path)

# 如果包里有动态库，把它们所在目录加入 LD_LIBRARY_PATH
lib_dirs = sorted({
    str(p.parent)
    for p in PREBUILT.rglob("*.so*")
    if p.is_file()
})

old_ld = os.environ.get("LD_LIBRARY_PATH", "")
ld_parts = lib_dirs + ([old_ld] if old_ld else [])

if ld_parts:
    os.environ["LD_LIBRARY_PATH"] = ":".join(ld_parts)

print("✓ sd-server:", SD_SERVER)

# 5. 确认二进制可运行
subprocess.run([
    SD_SERVER,
    "-h"
], check=False)


## 4. 下载 Z-Image-Turbo GGUF + Qwen3-4B GGUF + VAE

默认选 **Q4_K + Q4_K_M**，优先保证 Kaggle T4 的显存余量和吞吐。

如之后确认显存富余，可以把 diffusion 改为 Q5_0 / Q6_K，把 text encoder 改为 Q5_K_M / Q6_K 做画质对照。


In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download
import os

MODEL_DIR = Path("/kaggle/working/zimage_gguf/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Diffusion GGUF
DIFFUSION = hf_hub_download(
    repo_id="leejet/Z-Image-Turbo-GGUF",
    filename="z_image_turbo-Q4_K.gguf",
    local_dir=MODEL_DIR,
)

# Qwen3 4B text encoder GGUF
LLM = hf_hub_download(
    repo_id="unsloth/Qwen3-4B-Instruct-2507-GGUF",
    filename="Qwen3-4B-Instruct-2507-Q4_K_M.gguf",
    local_dir=MODEL_DIR,
)

# Z-Image / FLUX VAE
VAE = hf_hub_download(
    repo_id="Comfy-Org/z_image_turbo",
    filename="split_files/vae/ae.safetensors",
    local_dir=MODEL_DIR,
)

def gib(path):
    return Path(path).stat().st_size / 1024**3

print(f"diffusion: {DIFFUSION} | {gib(DIFFUSION):.2f} GiB")
print(f"llm      : {LLM} | {gib(LLM):.2f} GiB")
print(f"vae      : {VAE} | {gib(VAE):.2f} GiB")
print(f"total    : {gib(DIFFUSION)+gib(LLM)+gib(VAE):.2f} GiB files")


## 5. 配置（Kaggle Inference Hub 模型路由）

本 Notebook 只领取 `z-image-turbo-gguf` 队列，不会与 SANA 抢任务。


In [ ]:
BASE = "https://ranran-sana.202820.xyz"
MODEL = "z-image-turbo-gguf"
PASSWORD = "wangran"

import uuid
WORKER_ID = f"zimage-{uuid.uuid4().hex[:8]}"
TASK_URL = f"{BASE}/task/next?model={MODEL}&worker_id={WORKER_ID}"
UPLOAD_URL = f"{BASE}/upload"
REGISTER_URL = f"{BASE}/worker/register"
HEARTBEAT_URL = f"{BASE}/worker/heartbeat"
FAIL_URL = f"{BASE}/task/fail"

PORTS = [12340, 12341]
Z_STEPS = 8
CFG_SCALE = 1.0
USE_REMOTE_STEPS = True
TASK_QUEUE_SIZE = 64
UPLOAD_QUEUE_SIZE = 64
IO_WORKERS = 4

print({"worker_id":WORKER_ID,"model":MODEL,"TASK_URL":TASK_URL,"UPLOAD_URL":UPLOAD_URL,"ports":PORTS,"steps":Z_STEPS,"cfg":CFG_SCALE})


## 6. 启动两个常驻 sd-server

每个进程只看见一张物理 GPU：

- server 0: `CUDA_VISIBLE_DEVICES=0` → 进程内部 `cuda0`
- server 1: `CUDA_VISIBLE_DEVICES=1` → 进程内部 `cuda0`

这避免了 Z-Image 的单次推理跨两张 T4；我们要的是**两个独立生成 worker 并行吞吐**。


In [ ]:
import os, subprocess, time
from pathlib import Path
import requests

assert "SD_SERVER" in globals(), "请先执行 stable-diffusion.cpp 编译 Cell"
assert Path(DIFFUSION).exists()
assert Path(LLM).exists()
assert Path(VAE).exists()

# Notebook 内重复运行时先关掉由本 notebook 保存的旧进程
if "SD_PROCS" in globals():
    for p in SD_PROCS:
        if p and p.poll() is None:
            p.terminate()
    time.sleep(1)

SD_PROCS = []
SD_LOGS = []

def tail(path, n=80):
    try:
        return "\n".join(Path(path).read_text(errors="ignore").splitlines()[-n:])
    except Exception:
        return ""

def wait_server(port, proc, log_path, timeout=600):
    url = f"http://127.0.0.1:{port}/v1/models"
    start = time.time()
    while time.time() - start < timeout:
        if proc.poll() is not None:
            raise RuntimeError(
                f"sd-server 提前退出，code={proc.returncode}\n\n{tail(log_path)}"
            )
        try:
            r = requests.get(url, timeout=2)
            if r.ok:
                print(f"✓ sd-server :{port} ready")
                return
        except requests.RequestException:
            pass
        time.sleep(1)
    raise TimeoutError(f"等待 sd-server :{port} 超时\n\n{tail(log_path)}")

def start_server(physical_gpu, port):
    env = os.environ.copy()
    # 每个进程只暴露一张物理 T4；该进程内部它就是 cuda0
    env["CUDA_VISIBLE_DEVICES"] = str(physical_gpu)

    log_path = f"/kaggle/working/zimage_gpu{physical_gpu}.log"
    log = open(log_path, "w", buffering=1)

    cmd = [
        SD_SERVER,
        "--diffusion-model", DIFFUSION,
        "--vae", VAE,
        "--llm", LLM,
        "--backend", "cuda0",
        "--diffusion-fa",
        "--cfg-scale", str(CFG_SCALE),
        "--listen-ip", "127.0.0.1",
        "--listen-port", str(port),
        "-v",
    ]

    print("START:", " ".join(cmd))
    proc = subprocess.Popen(
        cmd,
        env=env,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    return proc, log_path, log

# 顺序启动，避免两边同时从磁盘加载模型导致无意义的 IO 争抢
for gpu, port in enumerate(PORTS):
    proc, log_path, log = start_server(gpu, port)
    SD_PROCS.append(proc)
    SD_LOGS.append((log_path, log))
    wait_server(port, proc, log_path)

print("✓ GPU0 + GPU1 两个常驻 Z-Image server 已启动")
subprocess.run(["nvidia-smi"], check=False)


## 7. 可选：双卡 warm-up

`sd-server` 可能会在第一次真正生成时才完成部分 lazy weight loading。

此 Cell 用两个线程同时让 GPU0 / GPU1 各做一次很小的 warm-up。失败不会中断后续 Cell，会打印对应 server 日志。


In [ ]:
import concurrent.futures, requests, base64, time

def warm_one(gpu, port):
    payload = {
        "prompt": "a simple red cube on a neutral background",
        "negative_prompt": "",
        "width": 256,
        "height": 256,
        "steps": 1,
        "cfg_scale": CFG_SCALE,
        "seed": 1000 + gpu,
        "batch_size": 1,
    }
    t = time.perf_counter()
    r = requests.post(
        f"http://127.0.0.1:{port}/sdapi/v1/txt2img",
        json=payload,
        timeout=(10, 900),
    )
    r.raise_for_status()
    x = r.json()
    assert x.get("images"), x
    return gpu, time.perf_counter() - t, len(base64.b64decode(x["images"][0]))

with concurrent.futures.ThreadPoolExecutor(max_workers=2) as ex:
    futures = [ex.submit(warm_one, i, p) for i, p in enumerate(PORTS)]
    for f in concurrent.futures.as_completed(futures):
        try:
            gpu, sec, size = f.result()
            print(f"✓ GPU{gpu} warm | {sec:.2f}s | {size/1024:.1f} KiB")
        except Exception as e:
            print("warm-up failed:", repr(e))

subprocess.run(["nvidia-smi"], check=False)


## 8. 可选：单张 1024×1024 冒烟测试

直接请求 GPU0 的本地 `sd-server`，不经过 Windows / Cloudflare。


In [ ]:
import requests, base64, io, time
from PIL import Image
from IPython.display import display

prompt = "A cinematic futuristic Tokyo alley after rain, subtle neon reflections, realistic photography, intricate details, no people"

payload = {
    "prompt": prompt,
    "negative_prompt": "",
    "width": 1024,
    "height": 1024,
    "steps": Z_STEPS,
    "cfg_scale": CFG_SCALE,
    "seed": 20260819,
    "batch_size": 1,
}

t = time.perf_counter()
r = requests.post(
    f"http://127.0.0.1:{PORTS[0]}/sdapi/v1/txt2img",
    json=payload,
    timeout=(10, 900),
)
r.raise_for_status()

data = r.json()
png = base64.b64decode(data["images"][0])
img = Image.open(io.BytesIO(png))
print(f"✓ smoke test: {img.size} | {time.perf_counter()-t:.2f}s")
display(img)


## 9. 启动 Worker：模型路由 + 注册 + 心跳 + AES-GCM 回传

双 T4 各自绑定一个常驻 `sd-server`；任务失败会报告本地 Hub，并按最大尝试次数决定是否回队列。


In [ ]:
import os, time, queue, threading, asyncio, io, hashlib, base64
import requests, aiohttp
from PIL import Image
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

KEY = hashlib.sha256(PASSWORD.encode()).digest()
tasks = queue.Queue(maxsize=TASK_QUEUE_SIZE)
uploads = queue.Queue(maxsize=UPLOAD_QUEUE_SIZE)
STOP = threading.Event()


def png_to_webp(png_bytes):
    with Image.open(io.BytesIO(png_bytes)) as im:
        im = im.convert("RGB"); b = io.BytesIO(); im.save(b, "WEBP", quality=90, method=4); return b.getvalue()


def encrypt(data):
    nonce = os.urandom(12); return nonce + AESGCM(KEY).encrypt(nonce, data, None)


async def upload_image(x, session):
    try:
        webp = await asyncio.to_thread(png_to_webp, x["png"])
        encrypted = await asyncio.to_thread(encrypt, webp)
        last = None
        for attempt in range(3):
            form = aiohttp.FormData()
            form.add_field("file", encrypted, filename=f'{x["id"]:04d}.bin', content_type="application/octet-stream")
            for k in ("id", "model", "worker_id", "gpu", "seed", "prompt", "seconds", "steps"):
                form.add_field(k, str(x[k]))
            try:
                async with session.post(UPLOAD_URL, data=form) as r:
                    if r.status >= 400: raise RuntimeError(f"HTTP {r.status}: {await r.text()}")
                last = None; break
            except Exception as e:
                last = e
                if attempt < 2: await asyncio.sleep(2 ** attempt)
        if last: raise last
        print(f'↑ #{x["id"]:03d} | GPU{x["gpu"]} → PC')
    except Exception as e: print(f'! Upload #{x.get("id")}: {e}')
    finally: x.pop("png", None); uploads.task_done()


async def upload_worker(session):
    while True:
        x = await asyncio.to_thread(uploads.get)
        if x is None: uploads.task_done(); break
        await upload_image(x, session)


async def register_worker(session):
    gpu_names=[]
    try:
        for i in range(2):
            gpu_names.append(os.popen(f"nvidia-smi --query-gpu=name --format=csv,noheader -i {i}").read().strip() or f"GPU{i}")
    except Exception: gpu_names=["GPU0","GPU1"]
    payload={"worker_id":WORKER_ID,"model":MODEL,"gpus":gpu_names,"runtime":"stable-diffusion.cpp","concurrency":2,"meta":{"ports":PORTS,"cfg":CFG_SCALE}}
    async with session.post(REGISTER_URL,json=payload) as r:
        if r.status>=400: raise RuntimeError(f"register HTTP {r.status}: {await r.text()}")
    print(f"✓ registered {WORKER_ID} -> {MODEL}")


async def heartbeat(session):
    while not STOP.is_set():
        try:
            async with session.post(HEARTBEAT_URL,json={"worker_id":WORKER_ID,"local_queue":tasks.qsize(),"upload_queue":uploads.qsize()}) as r:
                if r.status==404: await register_worker(session)
        except Exception as e: print(f"! Heartbeat: {e}")
        await asyncio.sleep(10)


async def task_poller(session):
    while not STOP.is_set():
        try:
            async with session.get(TASK_URL) as r:
                if r.status == 204: continue
                if r.status >= 400: raise RuntimeError(f"HTTP {r.status}: {await r.text()}")
                x = await r.json(); await asyncio.to_thread(tasks.put, x); print(f'↓ #{x["id"]:03d} | {x["prompt"][:70]}')
        except Exception as e:
            if not STOP.is_set(): print(f"! Poller: {e}"); await asyncio.sleep(2)


async def network_main():
    timeout = aiohttp.ClientTimeout(total=40)
    headers = {"Authorization": f"Bearer {PASSWORD}"}
    async with aiohttp.ClientSession(headers=headers, timeout=timeout) as session:
        await register_worker(session)
        await asyncio.gather(task_poller(session),heartbeat(session),*(upload_worker(session) for _ in range(IO_WORKERS)))


def run_network(): asyncio.run(network_main())


def report_fail(task_id,error):
    try:
        requests.post(FAIL_URL,headers={"Authorization":f"Bearer {PASSWORD}"},json={"id":task_id,"error":str(error),"requeue":True},timeout=15)
    except Exception as e: print(f"! fail report #{task_id}: {e}")


def generate_on_server(gpu, port, x):
    steps = int(x.get("steps", Z_STEPS)) if USE_REMOTE_STEPS else Z_STEPS
    payload = {"prompt":x["prompt"],"negative_prompt":"","width":int(x.get("width",1024)),"height":int(x.get("height",1024)),"steps":steps,"cfg_scale":CFG_SCALE,"seed":int(x["seed"]),"batch_size":1}
    with requests.Session() as s:
        r=s.post(f"http://127.0.0.1:{port}/sdapi/v1/txt2img",json=payload,timeout=(10,900)); r.raise_for_status(); data=r.json()
    if not data.get("images"): raise RuntimeError(f"sd-server 返回中没有 images: {data}")
    return base64.b64decode(data["images"][0]), steps


def gpu_worker(gpu, port):
    while True:
        x=tasks.get()
        if x is None: tasks.task_done(); break
        start=time.perf_counter()
        try:
            last_error=None
            for attempt in range(2):
                try: png,steps=generate_on_server(gpu,port,x); last_error=None; break
                except Exception as e:
                    last_error=e
                    if attempt==0: print(f'↻ #{x["id"]:03d} | GPU{gpu} retry | {e}'); time.sleep(1)
            if last_error is not None: raise last_error
            sec=round(time.perf_counter()-start,3)
            uploads.put({"id":x["id"],"model":MODEL,"worker_id":WORKER_ID,"gpu":gpu,"seed":x["seed"],"prompt":x["prompt"],"seconds":sec,"steps":steps,"png":png})
            print(f'✓ #{x["id"]:03d} | GPU{gpu} | {x.get("width",1024)}x{x.get("height",1024)} | {steps} steps | {sec:.2f}s')
        except Exception as e:
            print(f'✗ #{x.get("id")} | GPU{gpu} | {e}'); report_fail(x["id"],e)
        finally: tasks.task_done()


network_thread=threading.Thread(target=run_network,daemon=True); network_thread.start()
gpu_workers=[threading.Thread(target=gpu_worker,args=(0,PORTS[0]),daemon=True),threading.Thread(target=gpu_worker,args=(1,PORTS[1]),daemon=True)]
for w in gpu_workers: w.start()
print(f"✓ Z-Image Worker {WORKER_ID} 已启动 | GPU0:12340 + GPU1:12341 | Q4_K | 等待 {MODEL} Prompt...")


## 10. 状态检查


In [ ]:
print("STOP:", STOP.is_set())
print("task queue:", tasks.qsize(), "/", TASK_QUEUE_SIZE)
print("upload queue:", uploads.qsize(), "/", UPLOAD_QUEUE_SIZE)

for i, p in enumerate(SD_PROCS):
    print(f"sd-server GPU{i}:", "running" if p.poll() is None else f"dead({p.returncode})")

subprocess.run(["nvidia-smi"], check=False)


## 11. 停止 Dispatcher（不杀 sd-server）

执行后不再继续领取新的远程 Prompt；等待中的 worker 是 daemon thread，通常直接重启 kernel 最干净。

如果准备彻底释放显存，再执行下一 Cell。


In [ ]:
STOP.set()
print("✓ 已设置 STOP；task poller 将停止继续领取任务")


## 12. 停止两个 sd-server / 释放模型显存


In [ ]:
import time

if "SD_PROCS" in globals():
    for i, p in enumerate(SD_PROCS):
        if p.poll() is None:
            p.terminate()

    deadline = time.time() + 10
    for p in SD_PROCS:
        remain = max(0, deadline - time.time())
        try:
            p.wait(timeout=remain)
        except Exception:
            if p.poll() is None:
                p.kill()

if "SD_LOGS" in globals():
    for _, log in SD_LOGS:
        try:
            log.close()
        except Exception:
            pass

print("✓ sd-server 已停止")
subprocess.run(["nvidia-smi"], check=False)


## 13. 故障排查

### A. `sd-server` 启动后立刻退出
查看：

```python
print(open("/kaggle/working/zimage_gpu0.log", errors="ignore").read()[-10000:])
print(open("/kaggle/working/zimage_gpu1.log", errors="ignore").read()[-10000:])
```

### B. 显存不足
按优先级处理：

1. diffusion 从 `Q4_K` 降为 `Q3_K`
2. text encoder 从 `Q4_K_M` 降为 `Q3_K_M`
3. 再考虑 `--offload-to-cpu`
4. 再考虑 `--max-vram`

不要优先把两张 T4 合并做 `diffusion=cuda0&cuda1`。本 notebook 的目标是**双实例并行吞吐**。

### C. 本地 UI 仍显示 SANA / steps=2
这是你的 Windows `recv.py - v2` 页面文字，没有影响任务协议。

本 notebook 默认：

```python
USE_REMOTE_STEPS = False
Z_STEPS = 8
```

所以即使本地任务里传 `steps=2`，实际 Z-Image 仍跑 8 steps。
